In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [3]:
LEAVE_OUT_POINTS = {
    "lowData" : Point(104.2833333, 52.3),
    "highData" : Point(7.58371693, 47.54260867),
    "aridData" : Point(-2.17, 30.13),
    "northData" : Point(-105.117, 69.1),
    "equitData" : Point(6.72, 0.38),
    "southernData" : Point(-48.06, -22.66),
    "antData" : Point(-68.13, -67.57),
    "lakeWoods" : Point(-93.72, 49.67),
    "tibetPlat" : Point(91.133, 29.7),
    "ethiopiaHigh" : Point(39.77, 12.542),
    "hotWet" : Point(72.82, 18.96)
}

In [4]:
# Load the GNIP dataset
gnipDF = pd.read_csv("Data/GNIP/GNIP_Cleaned (2025-07-22).csv")

# Convert to GeoDataFrame
gnipGDF = gpd.GeoDataFrame(gnipDF, geometry=gpd.points_from_xy(gnipDF.Lon, gnipDF.Lat))
gnipGDF.head()

,Lat,Lon,Alt,Date,H2,O18,geometry
0,-78.583331,-85.416667,3687.0,1990-01-01T00:00:00.0000000+01:00,-314.6,-40.09,POINT (-85.41667 -78.58333)
1,-75.583333,-20.566667,30.0,1965-07-15T00:00:00.0000000+02:00,-188.8,-22.12,POINT (-20.56667 -75.58333)
2,-75.583333,-20.566667,30.0,1965-08-15T00:00:00.0000000+02:00,-195.7,-24.31,POINT (-20.56667 -75.58333)
3,-75.583333,-20.566667,30.0,1965-09-15T00:00:00.0000000+02:00,-155.7,-19.11,POINT (-20.56667 -75.58333)
4,-75.583333,-20.566667,30.0,1965-10-15T00:00:00.0000000+02:00,-166.7,-19.70,POINT (-20.56667 -75.58333)


In [5]:
# Pull out the leave out points
leaveOutGDF = gpd.GeoDataFrame(
    LEAVE_OUT_POINTS.items(),
    columns=["Label", "geometry"],
    geometry="geometry",
    crs="EPSG:4326"
)

In [6]:
# Test a move out point and see if it matches any GNIP points
testPoiny = leaveOutGDF.iloc[0].geometry
matchedPoints = gnipGDF[gnipGDF.geom_equals_exact(testPoiny, tolerance=0.001)]
matchedPoints.geometry.unique()

<GeometryArray>
[<POINT (104.283 52.3)>]
Length: 1, dtype: geometry

In [7]:
tempGDF = gpd.GeoDataFrame(columns=gnipGDF.columns)
for point in LEAVE_OUT_POINTS.values():
    tempGDF = pd.concat([tempGDF, gnipGDF[gnipGDF.geom_equals_exact(point, tolerance=0.1)]])

C:\Users\jaxton.gray\AppData\Local\Temp\ipykernel_15760\1603137105.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tempGDF = pd.concat([tempGDF, gnipGDF[gnipGDF.geom_equals_exact(point, tolerance=0.1)]])


In [9]:
gnipGDF[(gnipGDF.Lat >= 20) & (gnipGDF.Lat <= 35) & (gnipGDF.Lon >= -2) & (gnipGDF.Lon <= 10)].geometry.unique()

<GeometryArray>
[<POINT (5.52 22.78)>,  <POINT (5.6 23.27)>,  <POINT (5.4 31.92)>,
 <POINT (7.86 33.88)>]
Length: 4, dtype: geometry

In [ ]:
# Issue is that the aridData point does not have any H2 data, so when it was cleaned all those points were removed. 
# Find new point close to it with H2 data
# Probably point POINT (5.6 23.27)

In [56]:
test = gnipGDF[gnipGDF.geom_equals_exact(LEAVE_OUT_POINTS["aridData"], tolerance=1)]
test
#test.x, test.y

,Lat,Lon,Alt,Date,H2,O18,geometry


In [44]:
tempGDF.geometry.unique()

<GeometryArray>
[  <POINT (104.283 52.3)>,   <POINT (7.584 47.543)>,  <POINT (-105.117 69.1)>,
      <POINT (6.72 0.38)>, <POINT (-48.06 -22.664)>,  <POINT (-68.13 -67.57)>,
   <POINT (-93.72 49.67)>,    <POINT (91.133 29.7)>,   <POINT (39.77 12.542)>]
Length: 9, dtype: geometry